In [ ]:
# Setup cell. Canonical text: paste this file into the first code cell of every stage notebook.
STAGE = 1  # the only line that differs between notebooks

import contextlib
import json
import os
import shutil
import subprocess
import sys
import tarfile
import urllib.request
from pathlib import Path

ON_COLAB = "google.colab" in sys.modules
OBJECT = "my-object"  # folder name under real-to-sim-twin/ for your capture
RECORDING_ID = "real-to-sim-twin"
REFERENCE_URL = "https://artifacts.aegeanai.com/projects/real-to-sim-twin/reference"

RERUN = "rerun-sdk[notebook]==0.41.0"
PACKAGES = {
    1: ["pycolmap==4.2.1"],
    2: ["git+https://github.com/facebookresearch/sam2@2b90b9f", "huggingface_hub", "gsplat==1.5.3", "ninja",
        "git+https://github.com/rmbrualla/pycolmap@cc7ea4b"],
    3: ["gsplat==1.5.3", "ninja", "open3d==0.20.0", "git+https://github.com/rmbrualla/pycolmap@cc7ea4b"],
    4: ["diffusers==0.42.1", "transformers==5.21.0", "xatlas==0.0.11", "open3d==0.20.0", "pycolmap==4.2.1"],
    5: ["trimesh==5.2.0", "coacd==1.0.14"],
    6: ["newton[sim,importers]==1.7.0", "usd-core==26.8", "trimesh==5.2.0", "lpips==0.1.4", "gsplat==1.5.3", "ninja",
        "pycolmap==4.2.1"],
}
# Built against the installed torch, so pip must not isolate the build.
NO_BUILD_ISOLATION = {6: ["git+https://github.com/NVlabs/nvdiffrast@v0.4.0"]}


def _pip(*packages):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *packages], check=True)


def _patch_pycolmap_fork():
    # The fork breaks on numpy 2: np.uint64(-1) raises OverflowError.
    import importlib
    import importlib.util
    importlib.invalidate_caches()  # the package was installed by pip in this process
    root = Path(importlib.util.find_spec("pycolmap").submodule_search_locations[0])
    for f in root.rglob("*.py"):
        text = f.read_text()
        if "np.uint64(-1)" in text:
            f.write_text(text.replace("np.uint64(-1)", "np.uint64(2**64-1)"))


if ON_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    _pip(RERUN, *PACKAGES[STAGE])
    if STAGE in NO_BUILD_ISOLATION:
        _pip("--no-build-isolation", *NO_BUILD_ISOLATION[STAGE])
    if STAGE in (2, 3):  # stages 2 and 3 read stage1/sparse/0 with the pycolmap fork, never the official wheel
        _patch_pycolmap_fork()
    PROJECT = Path(f"/content/drive/MyDrive/real-to-sim-twin/{OBJECT}")
else:
    if "R2S_PROJECT" not in os.environ:
        raise RuntimeError("Off Colab, set the R2S_PROJECT environment variable to the project folder.")
    PROJECT = Path(os.environ["R2S_PROJECT"])
PROJECT.mkdir(parents=True, exist_ok=True)


@contextlib.contextmanager
def atomic_dir(path):
    """Yield path.tmp; on success it replaces path, on failure path is untouched."""
    path = Path(path)
    tmp, old = path.with_name(path.name + ".tmp"), path.with_name(path.name + ".old")
    shutil.rmtree(tmp, ignore_errors=True)
    if old.exists() and not path.exists():
        old.rename(path)  # a crash between the two renames left the only good copy in .old
    shutil.rmtree(old, ignore_errors=True)
    tmp.mkdir(parents=True)
    try:
        yield tmp
        if path.exists():
            path.rename(old)
        tmp.rename(path)
    except BaseException:
        shutil.rmtree(tmp, ignore_errors=True)
        if old.exists() and not path.exists():
            old.rename(path)
        raise
    shutil.rmtree(old, ignore_errors=True)


def on_reference(stage):
    """True if this stage runs on the reference object.

    The reference archives mark each stage folder with an empty .reference file, and a stage
    run on the reference marks the folder it writes. So the answer comes from the stage's own
    input folder (capture/ for stage 1, stage<N-1>/ otherwise), not from ground_truth/, which
    stays in the project after any fallback. For stage 1 a marked capture/ that also holds files
    of your own (own_capture_files) is not the reference.
    """
    prev = "capture" if stage == 1 else f"stage{stage - 1}"
    if not (PROJECT / prev / ".reference").is_file():
        return False
    return not (stage == 1 and own_capture_files(PROJECT / "capture"))


VIDEO_SUFFIXES = (".mp4", ".mov", ".m4v", ".avi", ".mkv", ".webm")


def own_capture_files(capture) -> list:
    """Files in capture/ that the reference capture never holds, so they must be yours.

    The reference capture has no board.json, and its only video is video.mp4, extracted at the same
    time as its .reference marker. So board.json, a video under another name (an iPhone's .mov, say)
    or a video changed more than a minute after the marker means that you added your own capture to
    a folder where a reference run left its files. board.json is the signal that does not depend on
    file times; your own capture always needs it for the metric scale.
    """
    capture = Path(capture)
    marker = capture / ".reference"
    since = marker.stat().st_mtime if marker.exists() else float("-inf")
    own = ["board.json"] if (capture / "board.json").exists() else []
    for p in sorted(capture.glob("*")):
        if p.suffix.lower() in VIDEO_SUFFIXES and (p.name != "video.mp4" or p.stat().st_mtime > since + 60):
            own.append(p.name)
    return own


def capture_video(capture):
    """The capture's video: video.mp4, or the one video file of another common phone format."""
    capture = Path(capture)
    if (capture / "video.mp4").exists():
        return capture / "video.mp4"
    videos = [p for p in sorted(capture.glob("*")) if p.suffix.lower() in VIDEO_SUFFIXES]
    if len(videos) != 1:
        raise FileNotFoundError(f"capture/ needs exactly one video (video.mp4, or one .mov, .m4v, .avi, .mkv or "
                                f".webm file); it holds {[p.name for p in videos] or 'none'}")
    return videos[0]


def _extract_reference(archive, names=None):
    """Download a reference archive and move its top-level folders (or only `names`) into PROJECT.

    A non-empty folder of yours that a reference folder replaces is kept as <name>.yours (an
    older .yours is replaced); ground_truth/ is replaced without a copy. Nothing moves until the
    archive is fully extracted.
    """
    base = os.environ.get("R2S_REFERENCE_URL", REFERENCE_URL).rstrip("/")
    tmp = PROJECT / f".{archive}.part"
    req = urllib.request.Request(f"{base}/{archive}", headers={"User-Agent": "real-to-sim-twin"})
    with urllib.request.urlopen(req) as r, open(tmp, "wb") as f:
        shutil.copyfileobj(r, f)
    scratch = PROJECT / ".extract"
    shutil.rmtree(scratch, ignore_errors=True)
    try:
        with tarfile.open(tmp) as tar:
            tar.extractall(scratch, filter="data")
        for item in scratch.iterdir():  # move into place only after a complete extraction
            if names is not None and item.name not in names:
                continue
            target = PROJECT / item.name
            if target.is_dir() and any(target.iterdir()) and item.name != "ground_truth":
                yours = PROJECT / f"{item.name}.yours"
                shutil.rmtree(yours, ignore_errors=True)
                target.rename(yours)
                print(f"your {item.name}/ is kept as {yours.name}/ and replaced by the reference {item.name}/.")
            elif target.is_dir():
                shutil.rmtree(target)
            elif target.exists():
                target.unlink()
            item.rename(target)
    finally:
        shutil.rmtree(scratch, ignore_errors=True)
        tmp.unlink(missing_ok=True)


def ensure_reference_capture(stage):
    """Stages 5 and 6 on the reference object use the reference capture/ too. True if it fetched it.

    When the stage's input folder is the reference object's (on_reference(stage), which follows
    the .reference markers from any earlier fallback) but capture/ is not (no capture/.reference),
    your measurements in capture/ belong to a different object than the mesh. The reference
    capture/ (physical.json, and the second-light files for stage 6) is then taken from the
    previous stage's archive, and yours is kept as capture.yours/.
    """
    if stage not in (5, 6) or not on_reference(stage) or (PROJECT / "capture" / ".reference").is_file():
        return False
    archive = f"stage{stage - 1}.tar.gz"
    print(f"stage{stage - 1}/ holds the reference object, so this stage uses the reference capture/ ({archive}).")
    _extract_reference(archive, names={"capture"})
    return True


def ensure_inputs(stage):
    """True if it fell back to the reference object, False if your inputs were present.

    The reference archive for a stage is cumulative: it holds every earlier stage folder that
    later stages read, plus ground_truth/, so a fallback gives one consistent object (see
    _extract_reference for what happens to folders of yours). In stages 5 and 6 it then makes
    sure that a reference object also gets the reference capture/ (ensure_reference_capture).
    """
    prev = "capture" if stage == 1 else f"stage{stage - 1}"
    folder = PROJECT / prev
    fell_back = not (folder.is_dir() and any(folder.iterdir()))
    if fell_back:
        archive = f"stage{stage - 1}.tar.gz"
        print(f"{prev}/ is missing or empty: running on the reference object ({archive}).")
        _extract_reference(archive)
    ensure_reference_capture(stage)
    return fell_back


import rerun as rr

RRD_PATH = PROJECT / "rrd" / f"stage{STAGE}.rrd"
RRD_PATH.parent.mkdir(parents=True, exist_ok=True)
rr.init("real-to-sim-twin", recording_id=RECORDING_ID)
rr.save(RRD_PATH)
rr.log("world", rr.ViewCoordinates.RIGHT_HAND_Z_UP, static=True)


def show_rerun():
    """Show everything logged so far in an inline viewer. Call it last in the stage.

    The viewer replaces the file sink, which leaves stage<N>.rrd complete; the viewer only gets
    data logged after it attaches, so the file is replayed into it. Later log calls reach the
    viewer only, not the file.
    """
    rr.notebook_show()
    rr.log_file_from_path(RRD_PATH)

In this section you turn a phone video of your object into camera poses and a sparse point cloud measured in meters. Structure from motion recovers where each frame was taken and where the scene points are, but only up to an unknown scale: the same images fit a scene twice as large, seen from cameras twice as far away. A printed ChArUco board of known size, lying next to the object, removes that ambiguity. By the end of the section the world frame has its origin at the top-left corner of the printed board, +Z pointing up and lengths in meters, and every later stage works in it.

The poses come from incremental structure from motion as introduced with COLMAP (Schönberger and Frahm, CVPR 2016). The board is a ChArUco pattern, a chessboard whose white squares carry ArUco markers (Garrido-Jurado et al., Pattern Recognition 2014), so each corner can be identified even when part of the board is hidden. The projection model and triangulation follow Hartley and Zisserman [@Hartley2004-ci].

Runtime budget on Colab: about 6 to 7 minutes for 100 frames, on 2 CPU cores with no GPU.

## Capturing the object

The pose solver can only use what the video shows, so the capture decides most of what follows.

- Lock exposure and focus before you start recording. A change in either between frames makes the same surface look different and breaks feature matching.
- Lay the printed board flat on a table and stand the object on it or next to it. Keep the whole board in view in most frames.
- Walk three orbits around the object at three heights: low, level with its top, and looking down on it. Move slowly, so that consecutive frames overlap by about 70%.
- Nothing in the scene moves during the video: not the object, not the board, not the paper under them.
- Print the board at 100% scale. Then measure one square and one marker with a ruler and enter both values, in millimeters, in `capture/board.json`:

```json
{"squares_x": 7, "squares_y": 5, "square_mm_measured": 36.0, "marker_mm_measured": 27.0, "dictionary": "DICT_5X5_100"}
```

The video goes in `capture/` as `video.mp4`. A phone's own format, such as an iPhone's `.mov`, also works when it is the only video there.

If you ran a stage on the reference object before filming, the reference files are still in `capture/`. Put your capture in a fresh project folder instead (change `OBJECT` in the setup cell), or first delete the reference files from `capture/`, including the hidden `.reference` file and the carton's `physical.json`. Stage 1 stops with this advice when it finds your files next to the reference ones.

In [ ]:
import json
import shutil
import time

import numpy as np

T0 = time.time()
FALLBACK = ensure_inputs(STAGE)
CAPTURE = PROJECT / "capture"
REFERENCE = on_reference(STAGE)  # the reference capture/ carries a .reference marker
if not REFERENCE and (CAPTURE / ".reference").exists():
    raise RuntimeError(
        f"capture/ holds your own files ({', '.join(own_capture_files(CAPTURE))}) next to the files an earlier run "
        "on the reference object left there: the hidden .reference marker and the carton's physical.json. Delete "
        "capture/.reference and capture/physical.json (write your own physical.json before stage 5), or put your "
        "capture in a fresh project folder by changing OBJECT in the setup cell, then run this cell again.")
WORK = (Path("/content/work") if ON_COLAB else PROJECT / ".work") / "stage1"
shutil.rmtree(WORK, ignore_errors=True)
WORK.mkdir(parents=True)
SCAN_OBJECT = json.loads((PROJECT / "ground_truth" / "scan.json").read_text())["object"] if REFERENCE else None
print("object:", f"reference ({SCAN_OBJECT})" if REFERENCE else OBJECT)
print("capture/:", sorted(p.name for p in CAPTURE.iterdir()))

## Choosing the frames

A 60 s video at 30 frames per second has 1,800 frames. Neighboring frames are nearly identical, and some are smeared by hand motion. You keep one frame per window of 0.4 s (2.5 frames per second), the sharpest one in the window. The sharpness score is the variance of the Laplacian of the gray image $I$:

$$
L(x, y) = \frac{\partial^2 I}{\partial x^2} + \frac{\partial^2 I}{\partial y^2}, \qquad S = \operatorname{Var}_{x,y} L(x, y).
$$

Motion blur acts as a low-pass filter. It removes the high spatial frequencies, which is where the Laplacian is large, so a blurred frame has a small $S$. The score also depends on how much texture the frame shows, so frames are compared only within a window, where the view barely changes.

Every tenth kept frame is held out. The pose solver still uses it, so it gets a camera pose, but later stages never train on it. In stage 6 you compare renders of your twin with these photos, which the twin has never seen.

Write `extract_frames`. Name each kept frame after its index in the video (`f00042.jpg`) and cap its long side at 1920 pixels.

In [ ]:
import cv2
import numpy as np
from pathlib import Path


def extract_frames(video: Path, fps: float = 2.5, keep: int = 300, out: Path | None = None) -> list[Path]:
    """Keep the sharpest frame in each window of 1/fps seconds, at most `keep` frames.

    Score each frame by the variance of its Laplacian. Save the kept frames as
    out/f<index>.jpg, where <index> is the frame number in the video (f00042.jpg), with the
    long side capped at 1920 pixels. Return the saved paths in video order.
    """
    raise NotImplementedError

In [ ]:
frames = extract_frames(capture_video(CAPTURE), fps=2.5, keep=300, out=WORK / "frames")
heldout = frames[9::10]  # every tenth kept frame; later stages never train on these
print(f"{len(frames)} frames kept, {len(heldout)} held out")
assert len(frames) >= 30, "fewer than 30 frames kept: is the video shorter than 12 s?"

In [ ]:
import io

import matplotlib.pyplot as plt
from IPython.display import Image, display

show = frames[:: max(1, len(frames) // 8)][:8]
fig, axes = plt.subplots(2, 4, figsize=(14, 4.6))
for ax, p in zip(axes.ravel(), show):
    img = cv2.imread(str(p))
    small = cv2.resize(img, None, fx=0.5, fy=0.5, interpolation=cv2.INTER_AREA)
    s = cv2.Laplacian(cv2.cvtColor(small, cv2.COLOR_BGR2GRAY), cv2.CV_64F).var()
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{p.name}  sharpness {s:.0f}" + ("  held out" if p in heldout else ""), fontsize=9)
    ax.axis("off")
plt.tight_layout()
buf = io.BytesIO()
fig.savefig(buf, format="jpeg", dpi=80, pil_kwargs={"quality": 85})  # photos: JPEG keeps the page small
plt.close(fig)
display(Image(data=buf.getvalue()))

## Poses up to scale

A camera with intrinsics $K$, rotation $R$ and translation $t$ maps a scene point $X$ to the pixel $x$ through

$$
\lambda \begin{bmatrix} x \\ 1 \end{bmatrix} = K (R X + t), \qquad \lambda > 0.
$$

Structure from motion finds all $R_i$, $t_i$ and $X_j$ that minimize the reprojection error over every observation of point $j$ in frame $i$:

$$
\min_{\{R_i, t_i\}, \{X_j\}} \sum_{i,j} \left\| x_{ij} - \pi\!\left(K (R_i X_j + t_i)\right) \right\|^2,
$$

where $\pi$ divides by the third coordinate. Replace every $X_j$ by $s X_j$ and every $t_i$ by $s t_i$, for any $s > 0$. Then $R_i X_j + t_i$ becomes $s (R_i X_j + t_i)$, the factor cancels in the division, and every pixel stays where it was. The error does not change, so the images alone cannot tell a small scene from a large one. The solver fixes the scale by an arbitrary convention, and its unit is not a meter.

The solver below detects SIFT features on frames reduced to 960 pixels on the long side, keeps at most 4,096 per frame, matches each frame with its neighbors in the video, and then grows the model one frame at a time, refining all poses and points together as it goes.

In [ ]:
import pycolmap

pycolmap.logging.minloglevel = 2  # errors only
db = WORK / "database.db"
reader = pycolmap.ImageReaderOptions()
reader.camera_model = "SIMPLE_RADIAL"
extraction = pycolmap.FeatureExtractionOptions()
extraction.max_image_size = 960
extraction.sift.max_num_features = 4096
pycolmap.extract_features(db, WORK / "frames", camera_mode=pycolmap.CameraMode.SINGLE,
                          reader_options=reader, extraction_options=extraction, device=pycolmap.Device.cpu)
pairing = pycolmap.SequentialPairingOptions()
pairing.overlap, pairing.quadratic_overlap, pairing.loop_detection = 20, True, False
pycolmap.match_sequential(db, pairing_options=pairing, device=pycolmap.Device.cpu)
(WORK / "sparse").mkdir()
maps = pycolmap.incremental_mapping(db, WORK / "frames", WORK / "sparse")
model = max(maps.values(), key=lambda m: m.num_reg_images())
print(f"{len(maps)} model(s); the largest registers {model.num_reg_images()} of {len(frames)} frames, "
      f"{model.num_points3D()} points, mean reprojection error {model.compute_mean_reprojection_error():.2f} px")

## Scale from the board

The board supplies the missing length. Its corner $k$ sits at a known position $B_k$ on the board plane, in meters, computed from your measured square size. Triangulating each corner from the frames that see it, with the poses the solver found, gives the same corner $Y_k$ in model units. The two sets are related by a similarity,

$$
Y_k \approx s R B_k + t,
$$

which has a closed-form least-squares solution (Umeyama, IEEE TPAMI 1991). One model unit is then $1/s$ meters. The same fit locates the board plane, and you place the world so that the board is the plane $z = 0$ with +Z toward the cameras. The model written to `stage1/sparse/0` has this transform already applied, so its points and camera centers are in meters, and `scale.json` marks it with `"applied": true`. Its `metres_per_unit` and `world_from_model` record the original gauge of the solver and are never applied a second time. `board_reproj_px` is the mean distance, in pixels, between each detected corner and the fitted board corner projected into that frame.

A single global fit can hide disagreement between frames, so each frame also gets its own estimate. From one frame and the measured square alone, a perspective-n-point solve gives the board's pose in meters. Dividing the corner depths in meters by the same depths in model units gives that frame's scale $s_i$, and

$$
\text{scale\_cv} = \frac{\operatorname{std}_i(s_i)}{\operatorname{mean}_i(s_i)}
$$

is small when every frame sees the same rigid board. A bent board makes the frames disagree, and so do prints of different sizes in one scene. A printer that shrank the whole page by the same factor does not: every frame then agrees on a wrong size. Only your ruler catches that, which is why `board.json` holds the square you measured and not the nominal 36 mm. The check also looks at the extremes, $\text{scale\_spread} = (\max_i s_i - \min_i s_i) / \operatorname{median}_i s_i$. A scene mixing two prints that differ by 3% has a coefficient of variation of at most half that gap, 1.5%, while its spread is the full 3%.

Write `scale_from_board`.

In [ ]:
import cv2
import numpy as np
from pathlib import Path


def scale_from_board(model, frames, board: dict) -> dict:
    """Metric scale and a board-aligned world frame for a COLMAP model.

    1. Detect the ChArUco corners in every registered frame (cv2.aruco.CharucoDetector),
       with the board built from board.json (square and marker sizes in meters).
    2. Triangulate each corner seen in two or more frames, using the COLMAP poses.
    3. Fit a similarity between the board's corner positions (meters) and the
       triangulated corners (model units). Its scale gives metres_per_unit.
    4. For each frame, estimate the board pose in meters with solvePnP, and divide the
       corner depths in meters by the same depths in model units: one scale per frame.
    5. Build world_from_model (4x4, scale included) that puts the board plane at z = 0
       with +Z toward the cameras. The world frame is the OpenCV board frame with y and z
       flipped: its origin is the origin of getChessboardCorners(), the top-left corner of the
       printed board (marked on the PDF); +X runs along the board's top edge to the right, +Y
       up the page, away from the board across its top edge, and +Z up out of the board.

    Return {"metres_per_unit", "source": "board", "board_reproj_px", "scale_cv",
    "scale_spread", "board_frames", "world_from_model", "board_corners_world"}, where
    scale_cv is std/mean and scale_spread is (max - min)/median of the per-frame scales, and
    board_corners_world holds all (squares_x - 1) * (squares_y - 1) chessboard corners in
    CharucoBoard.getChessboardCorners() order, in world meters (stage 6 fits the board's pose in
    a photo to them).
    Raise RuntimeError when fewer than 3 frames give a board pose.
    """
    raise NotImplementedError

## Scale on the reference object

The reference object is OmniObject3D `boxed_beverage_001`, a 250 mL drink carton (CC BY 4.0). It was filmed on a calibration pattern of unknown size, so the board route is not available. The scale comes from the dataset's textured scan instead. The sparse points that fall inside the object's matting mask are aligned to the scan by a similarity, found by iterative closest points from a principal-axes start. The fit first holds the scale fixed and settles rotation and position. It then frees the scale and also pairs the nearest part of the scan back to the points, so the points cannot shrink onto a small patch of the scan, where every distance would be small. The table plane around the object, fitted to the remaining points, defines +Z. Views from above pin down how a long lying object sits only weakly, so the scan is then leveled in two turns. It turns about its long axis until its lowest points on either side of that axis sit at the same height, and about the horizontal axis across it until its lowest points at the two ends do: an object lying on a table touches it on both sides and at both ends. An object standing upright is left as fitted. The fit has one known limit. When the views cover only part of the object, as for a long object filmed mostly from above, the part of the scan nearest the fitted points includes surface no view saw. Those pairs pull the points outward, and the scale can come out several percent too large. In one synthetic test of a long object seen from above it came out 18% too large, and none of the checks in this section caught it, so passing them does not bound this error. Repeated runs on the same video also recover slightly different poses, so the metric scale varies from run to run, so far by up to about 0.4% on this object.

The scan's unit is millimeters. The dataset does not state it: it is inferred from the 250 mL label, since the scan's convex hull holds 324 mL if the unit is millimeters and a thousand times more or less for any other metric unit. The scale here is only as good as that inference. `scale.json` records `"source": "scan"`. On your own object the board sets the scale instead.

In [ ]:
import json

import numpy as np
from pathlib import Path
from scipy.spatial import cKDTree


def _umeyama(src, dst):
    mu_s, mu_d = src.mean(0), dst.mean(0)
    a, b = src - mu_s, dst - mu_d
    U, D, Vt = np.linalg.svd(b.T @ a / len(src))
    E = np.diag([1.0, 1.0, np.sign(np.linalg.det(U @ Vt))])
    R = U @ E @ Vt
    s = np.trace(np.diag(D) @ E) / (a**2).sum(1).mean()
    return s, R, mu_d - s * R @ mu_s


def load_obj_vertices(path):
    with open(path) as f:
        rows = [line.split()[1:4] for line in f if line.startswith("v ")]
    return np.array(rows, float)


def object_points(model, masks_dir, min_inside=0.8):
    """Sparse points whose observations fall inside the object's matting mask."""
    import cv2

    masks, keep = {}, []
    for pid, point in model.points3D.items():
        elements = point.track.elements
        if len(elements) < 3:
            continue
        inside = 0
        for el in elements:
            image = model.images[el.image_id]
            if image.name not in masks:
                masks[image.name] = cv2.imread(str(Path(masks_dir) / (Path(image.name).stem + ".png")),
                                               cv2.IMREAD_GRAYSCALE)
            m = masks[image.name]
            x, y = np.round(image.points2D[el.point2D_idx].xy).astype(int)
            inside += m is not None and 0 <= y < m.shape[0] and 0 <= x < m.shape[1] and m[y, x] > 127
        if inside >= min_inside * len(elements):
            keep.append(point.xyz)
    return np.array(keep)


def sim3_icp(src, dst_tree, dst, rigid_iters=40, scale_iters=200, trim=0.8, reverse=0.4):
    """Similarity ICP in two phases, from the 24 proper axis permutations of a PCA start.

    Phase 1 holds the scale at the PCA estimate and fits rotation and translation on the
    closest 80% of pairs, from every start; the start with the smallest trimmed distance wins.
    Phase 2 frees the scale with pairs in both directions: each sparse point to its nearest
    scan point (closest 80%), and the closest `reverse` share of the scan to its nearest
    fitted point. The reverse pairs stop the fit from shrinking onto a patch of the scan,
    which lowers every forward distance but leaves most of the surface unmatched. They are
    drawn from the whole scan, so on views that miss much of the object the closest share
    can include surface no camera saw, which pulls the scale up: by several percent or more
    on a long object seen from above, with no known bound.
    """
    def pca(p):
        mu = p.mean(0)
        _, sv, vt = np.linalg.svd(p - mu, full_matrices=False)
        return mu, vt, sv

    mu_s, v_s, sv_s = pca(src)
    mu_d, v_d, sv_d = pca(dst)
    s = np.sqrt((sv_d**2).sum() / len(dst) / ((sv_s**2).sum() / len(src)))
    best = None
    for perm in [(0, 1, 2), (0, 2, 1), (1, 0, 2), (1, 2, 0), (2, 0, 1), (2, 1, 0)]:
        for signs in [(1, 1, 1), (1, -1, -1), (-1, 1, -1), (-1, -1, 1), (-1, 1, 1), (1, -1, 1), (1, 1, -1), (-1, -1, -1)]:
            P = np.zeros((3, 3))
            P[range(3), perm] = signs
            R = v_d.T @ P @ v_s
            if np.linalg.det(R) < 0:
                continue
            t = mu_d - s * R @ mu_s
            for _ in range(rigid_iters):  # rigid: the scale stays at the PCA estimate
                d, j = dst_tree.query(src @ (s * R).T + t)
                k = d <= np.quantile(d, trim)
                _, R, _ = _umeyama(src[k], dst[j[k]])
                t = dst[j[k]].mean(0) - s * R @ src[k].mean(0)
            d, _ = dst_tree.query(src @ (s * R).T + t)
            score = np.mean(np.sort(d)[: int(trim * len(d))])
            if best is None or score < best[0]:
                best = (score, R, t)
    _, R, t = best
    rng = np.random.default_rng(0)
    rev = dst[rng.choice(len(dst), min(len(dst), 20 * len(src)), replace=False)]
    for _ in range(scale_iters):  # free scale, pairs in both directions
        fitted = src @ (s * R).T + t
        d, j = dst_tree.query(fitted)
        k = d <= np.quantile(d, trim)
        e, i = cKDTree(fitted).query(rev)  # scan points back to their nearest fitted point
        q = np.argsort(e)[: int(reverse * len(rev))]  # the part of the scan closest to the fitted points
        q = q[np.linspace(0, len(q) - 1, k.sum()).astype(int)]  # as many reverse pairs as forward pairs
        s_new, R, t = _umeyama(np.vstack([src[k], src[i[q]]]), np.vstack([dst[j[k]], rev[q]]))
        done = abs(s_new / s - 1) < 1e-5
        s = s_new
        if done:
            break
    d, _ = dst_tree.query(src @ (s * R).T + t)
    score = np.mean(np.sort(d)[: int(trim * len(d))])
    return score, s, R, t, d


def fit_extent_ratio(fitted, scan):
    """Spread of the fitted points along the scan's longest axis over the scan's own (2nd to 98th
    percentile). Near 1 for a good fit; a fit that shrank onto a patch of the scan gives a small value."""
    mu = scan.mean(0)
    axis = np.linalg.svd(scan - mu, full_matrices=False)[2][0]
    if len(fitted) < 2:
        return float("nan")  # nothing on the scan: the check fails it
    spread = lambda p: np.subtract(*np.percentile((p - mu) @ axis, [98, 2]))
    return float(spread(fitted) / spread(scan))


def table_frame(normal, long_axis=None):
    """Rotation (rows x, y, z) of a world with +Z = normal and x along a scan direction projected on
    the table: the scan's x, or, when the normal lies within 45 degrees of x (an object lying on its
    side), the object's long axis (or, without one, the scan axis most nearly parallel to the table)."""
    z = normal / np.linalg.norm(normal)
    axes = np.eye(3)
    if abs(z[0]) < np.sqrt(0.5):
        a = axes[0]
    elif long_axis is not None and abs(long_axis @ z) < np.sqrt(0.5):
        a = np.asarray(long_axis, float)
    else:
        a = axes[np.argmin(np.abs(axes @ z))]
    x = a - (a @ z) * z
    x /= np.linalg.norm(x)
    return np.stack([x, np.cross(z, x), z])


def _rotation(axis, angle):
    axis = axis / np.linalg.norm(axis)
    K = np.array([[0, -axis[2], axis[1]], [axis[2], 0, -axis[0]], [-axis[1], axis[0], 0]])
    return np.eye(3) + np.sin(angle) * K + (1 - np.cos(angle)) * K @ K


def _level(points, center, turn_axis, split_axis, plane_point, normal, max_deg, step_deg):
    """Angle about turn_axis at which the lowest points on either side of split_axis are equally high."""
    best = None
    for deg in np.arange(-max_deg, max_deg + step_deg / 2, step_deg):
        p = (points - center) @ _rotation(turn_axis, np.radians(deg)).T
        height = (p + center - plane_point) @ normal
        side = p @ split_axis > 0
        gap = abs(np.percentile(height[side], 0.1) - np.percentile(height[~side], 0.1))
        if best is None or gap < best[0] - 1e-9:
            best = (gap, np.radians(deg))
    return best[1]


def _table_axes(axis, normal):
    """The long axis laid flat on the table, and the horizontal axis across it."""
    flat = axis - (axis @ normal) * normal
    flat /= np.linalg.norm(flat)
    return flat, np.cross(normal, flat)


def rest_on_table(scan, axis, center, plane_point, normal, max_deg=15.0, step_deg=0.25):
    """Roll and tilt (radians) that make a lying scan rest level on the table plane.

    An object lying on a table touches it on both sides of its long axis and at both ends,
    so the lowest scan points on either side should sit at the same height (roll, about the
    long axis), and so should the lowest points at either end (tilt, about the horizontal
    axis across it). Returns (0.0, 0.0) when the long axis stands within 45 degrees of
    upright: such an object stands on its base, and the fit already fixes how it stands.
    """
    axis = axis / np.linalg.norm(axis)
    if abs(axis @ normal) > np.sqrt(0.5):
        return 0.0, 0.0
    flat, across = _table_axes(axis, normal)
    roll = _level(scan, center, axis, across, plane_point, normal, max_deg, step_deg)
    rolled = (scan - center) @ _rotation(axis, roll).T + center
    tilt = _level(rolled, center, across, flat, plane_point, normal, max_deg, step_deg)
    return roll, tilt


def level_rotation(axis, normal, roll, tilt):
    """The rotation of the scan that rest_on_table found: roll about the long axis, then tilt across it."""
    axis = axis / np.linalg.norm(axis)
    if roll == 0.0 and tilt == 0.0:
        return np.eye(3)
    return _rotation(_table_axes(axis, normal)[1], tilt) @ _rotation(axis, roll)


def turn_model(R, t, L, center):
    """The fit after turning the scan by L about center: the model turns by L^T instead."""
    return L.T @ R, L.T @ (t - center) + center


def table_plane(points):
    """Least-squares plane, refit twice after dropping the farthest tenth: (point on it, unit normal)."""
    for _ in range(3):
        mu = points.mean(0)
        normal = np.linalg.svd(points - mu)[2][-1]
        r = np.abs((points - mu) @ normal)
        points = points[r <= np.quantile(r, 0.9)]
    return mu, normal


def scale_from_scan(model, ground_truth: Path) -> dict:
    """Metric scale for the reference object, by aligning its sparse points to the scan.

    The scan's unit is millimeters. That is inferred (scan.json says from what), not documented
    by the dataset, so this scale is only as good as that inference.
    """
    ground_truth = Path(ground_truth)
    info = json.loads((ground_truth / "scan.json").read_text())
    scan = load_obj_vertices(ground_truth / info["mesh"])  # scan units (mm, inferred)
    rng = np.random.default_rng(0)
    scan = scan[rng.choice(len(scan), min(len(scan), 200_000), replace=False)]
    tree = cKDTree(scan)
    obj = object_points(model, ground_truth / "matting")
    score, s, R, t, d = sim3_icp(obj, tree, scan)
    mm_per_unit = s
    all_xyz = np.array([p.xyz for p in model.points3D.values()])
    cams_xyz = np.array([im.projection_center() for im in model.images.values() if im.has_pose])
    center = scan.mean(0)
    long_axis = np.linalg.svd(scan - center, full_matrices=False)[2][0]

    def table(R, t):
        # Up is the normal of the table plane, fitted to the sparse points near the object but
        # outside its mask, oriented toward the cameras.
        all_pts = all_xyz @ (s * R).T + t  # in scan mm
        near = all_pts[np.linalg.norm(all_pts - center, axis=1) < 250]
        d_obj, _ = tree.query(near)
        mu, normal = table_plane(near[d_obj > 5])
        if np.mean((cams_xyz @ (s * R).T + t - mu) @ normal) < 0:
            normal = -normal
        return mu, normal

    # Roll about the long axis and tilt across it: views from above fix both only weakly, so
    # rest the scan on the table. Turning the scan by Q is turning the model by Q^T.
    mu, normal = table(R, t)
    roll, tilt = rest_on_table(scan, long_axis, center, mu, normal)
    R, t = turn_model(R, t, level_rotation(long_axis, normal, roll, tilt), center)
    mu, normal = table(R, t)
    fitted = obj @ (s * R).T + t
    d, _ = tree.query(fitted)
    height = (scan - mu) @ normal

    # World: +Z = table normal, x along a scan direction projected on the table, origin on the
    # table under the scan's center, meters.
    world_from_scan_R = table_frame(normal, long_axis)
    origin = center - ((center - mu) @ normal) * normal
    world_from_scan = np.eye(4)
    world_from_scan[:3, :3] = world_from_scan_R * 0.001
    world_from_scan[:3, 3] = -world_from_scan_R @ origin * 0.001
    scan_from_model = np.eye(4)
    scan_from_model[:3, :3], scan_from_model[:3, 3] = s * R, t
    M = world_from_scan @ scan_from_model
    return {
        "metres_per_unit": float(mm_per_unit * 0.001),
        "source": "scan",
        "board_reproj_px": None,
        "scale_cv": None,
        "scan_fit_mm": float(np.median(d)),
        "scan_points_used": int(len(obj)),
        "fit_extent_ratio": fit_extent_ratio(fitted[d < 3], scan),  # inliers only: strays do not stretch it
        "scan_roll_deg": float(np.degrees(roll)),
        "scan_tilt_deg": float(np.degrees(tilt)),
        "scan_base_mm": float(np.percentile(height, 0.1)),
        "up_vs_scan_y_deg": float(np.degrees(np.arccos(abs(normal[1])))),
        "world_from_model": M.tolist(),
        "scan_from_world": np.linalg.inv(world_from_scan).tolist(),
    }

In [ ]:
centers = np.array([im.projection_center() for im in sorted(model.images.values(), key=lambda im: im.name)])
units_path = np.linalg.norm(np.diff(centers, axis=0), axis=1).sum()  # camera path length, model units
if REFERENCE:
    scale = scale_from_scan(model, PROJECT / "ground_truth")
else:
    board = json.loads((CAPTURE / "board.json").read_text())
    scale = scale_from_board(model, frames, board)
M = np.array(scale["world_from_model"])
model.transform(pycolmap.Sim3d(M[:3, :]))
print(f"1 model unit = {1000 * scale['metres_per_unit']:.2f} mm (source: {scale['source']})")
print(f"camera path: {units_path:.2f} model units = {units_path * scale['metres_per_unit']:.3f} m")
if scale["source"] == "board":
    print(f"board in {scale['board_frames']} frames, reprojection {scale['board_reproj_px']:.2f} px, "
          f"scale_cv {100 * scale['scale_cv']:.2f}%, spread {100 * scale['scale_spread']:.2f}%")
else:
    print(f"scan fit: median distance {scale['scan_fit_mm']:.2f} mm over {scale['scan_points_used']} object points, "
          f"spanning {scale['fit_extent_ratio']:.2f} of the scan's length")
    print(f"scan leveled on the table: rolled {scale['scan_roll_deg']:+.2f} and tilted {scale['scan_tilt_deg']:+.2f} "
          f"degrees; its base sits {scale['scan_base_mm']:+.1f} mm from the table plane")

## The scene in meters

The viewer shows the sparse points, a frustum and thumbnail for every registered frame, and the board corners in the world frame. On the reference object it shows the scan, drawn in gray, in place of the board. With a correct scale and world frame, the board corners lie on the printed pattern in the point cloud, the object stands upright on the plane $z = 0$, and the frustums sit at heights you can check against how you held the phone.

In [ ]:
def log_stage1(model, scale, project):
    rr.log("world/points", rr.Points3D([p.xyz for p in model.points3D.values()],
                                       colors=[p.color for p in model.points3D.values()], radii=0.0008), static=True)
    for image in sorted(model.images.values(), key=lambda im: im.name):
        cam = image.camera
        thumb = cv2.imread(str(WORK / "frames" / image.name))
        k = 320 / thumb.shape[1]
        thumb = cv2.resize(thumb, None, fx=k, fy=k, interpolation=cv2.INTER_AREA)
        world_from_cam = image.cam_from_world().inverse()
        path = f"world/cameras/{Path(image.name).stem}"
        rr.log(path, rr.Transform3D(translation=world_from_cam.translation,
                                    mat3x3=world_from_cam.rotation.matrix()), static=True)
        rr.log(path, rr.Pinhole(focal_length=cam.focal_length * k, width=thumb.shape[1], height=thumb.shape[0],
                                image_plane_distance=0.03), static=True)
        rr.log(path, rr.EncodedImage(contents=cv2.imencode(".jpg", thumb)[1].tobytes(), media_type="image/jpeg"),
               static=True)
    if "board_corners_world" in scale:
        rr.log("world/board_corners", rr.Points3D(scale["board_corners_world"], colors=[255, 120, 0], radii=0.002),
               static=True)
    if "scan_from_world" in scale:
        scan = load_obj_vertices(project / "ground_truth" / "scan" / "Scan.obj")[::25]
        T = np.linalg.inv(np.array(scale["scan_from_world"]))
        rr.log("world/ground_truth/scan", rr.Points3D(scan @ T[:3, :3].T + T[:3, 3], colors=[160, 160, 160],
                                                      radii=0.0005), static=True)


log_stage1(model, scale, PROJECT)

In [ ]:
show_rerun()

## Check

The check writes `stage1/` only when at least 80% of the kept frames are registered. Below that, the model covers only part of the orbit, and any scale fitted to it would be carried into every later stage. Recapture, or let the next stage fall back to the reference `stage1/`. On the reference object the check also requires 90% of the frames registered, a mean reprojection error under 1.5 px and a median distance to the scan under 3 mm. It also requires the fitted points that lie on the scan to span at least 0.7 times its length, because a fit that shrank onto a small patch of the scan passes the distance test (a fit that came out too large is not caught by this gate), and the scan's base to lie within 3 mm of the table plane, because the object rests on the table. On your own object the registration and reprojection gates become warnings, together with a warning when `scale_cv` or `scale_spread` exceeds 2%, which points to a rescaled print mixed into the scene or a bent board. When a reference gate fails, or the board fit gives no usable frames, nothing is written.

In [ ]:
import json
import shutil
from pathlib import Path


def write_stage1(out: Path, frames: list, model, scale: dict, heldout: list) -> None:
    """Write stage1/ in one step, or refuse when too few frames are registered."""
    registered = model.num_reg_images()
    fraction = registered / len(frames)
    if fraction < 0.8:
        raise RuntimeError(
            f"Only {registered} of {len(frames)} frames ({fraction:.0%}) are registered, below 80%. "
            "stage1/ and scale.json were not written. Recapture following the capture rules, or "
            "continue from the reference object: with stage1/ missing, the stage 2 setup cell "
            "downloads the reference stage1/.")
    keys = ["metres_per_unit", "source", "board_reproj_px", "scale_cv", "scale_spread", "board_frames",
            "scan_fit_mm", "fit_extent_ratio", "scan_roll_deg", "scan_tilt_deg", "scan_base_mm", "world_from_model", "scan_from_world",
            "board_corners_world"]
    scale = {k: scale[k] for k in keys if k in scale}
    scale["applied"] = True
    scale["note"] = ("sparse/0 is already in meters in the world frame (+Z up). metres_per_unit and "
                     "world_from_model record the transform already applied to the SfM model; do not apply them again.")
    with atomic_dir(out) as d:
        shutil.copytree(Path(frames[0]).parent, d / "frames")
        (d / "sparse" / "0").mkdir(parents=True)
        model.write(d / "sparse" / "0")
        (d / "scale.json").write_text(json.dumps(scale, indent=2))
        (d / "heldout.txt").write_text("".join(f"{Path(p).name}\n" for p in heldout))

In [ ]:
import math

LO_SPAN = 0.7  # fitted points on the scan over the scan's length, along its longest axis
LEVEL_LIMIT_DEG = 15.0  # the roll and tilt search range of rest_on_table


def stage1_problems(registered: int, total: int, reproj: float, scale: dict, reference: bool):
    """Return (failures, warnings) for the stage 1 check."""
    failures, warnings = [], []
    gates = []
    if registered / total < 0.9:
        gates.append(f"only {registered / total:.0%} of frames registered (want 90%)")
    if reproj >= 1.5:
        gates.append(f"mean reprojection error {reproj:.2f} px (want under 1.5 px)")
    fit = []  # a scan fit that shrank onto a patch, or sits rolled into the table, passes the distance gate
    if scale.get("source") == "scan":
        ratio, base = scale.get("fit_extent_ratio"), scale.get("scan_base_mm")
        if ratio is None:
            fit.append("the scan fit reported no fit_extent_ratio")
        elif not LO_SPAN <= ratio:  # points within 3 mm of the scan cannot reach far past it: this sees shrinking only
            fit.append(f"the fitted object points on the scan span {ratio:.2f} of its length (want at least {LO_SPAN}): "
                       "the scan fit shrank onto part of the scan")
        if base is None:
            fit.append("the scan fit reported no scan_base_mm")
        elif not abs(base) <= 3:
            fit.append(f"the scan's base sits {base:+.1f} mm from the table plane (want within 3 mm): "
                       "the object should rest on the table")
    if scale.get("source") == "scan" and max(abs(scale.get("scan_roll_deg", 0.0)),
                                             abs(scale.get("scan_tilt_deg", 0.0))) >= LEVEL_LIMIT_DEG:
        warnings.append(f"leveling stopped at its {LEVEL_LIMIT_DEG:.0f}-degree limit: the scan may need a larger turn")
    if reference:
        failures += gates + fit
        if not scale.get("scan_fit_mm", math.inf) <= 3:
            failures.append(f"sparse points sit {scale.get('scan_fit_mm')} mm from the scan (want under 3 mm)")
        return failures, warnings
    warnings += gates + fit
    if scale.get("source") == "scan":
        return failures, warnings
    for key in ["metres_per_unit", "scale_cv", "scale_spread", "board_reproj_px"]:
        value = scale.get(key)
        if value is None or not math.isfinite(value):
            failures.append(f"{key} is missing or not finite: the board fit failed")
    if failures:
        return failures, warnings
    if scale["scale_cv"] > 0.02:
        warnings.append(f"scale_cv {100 * scale['scale_cv']:.1f}% > 2%: the frames disagree on the board's size")
    if scale["scale_spread"] > 0.02:
        warnings.append(f"scale_spread {100 * scale['scale_spread']:.1f}% > 2%: the largest and smallest per-frame "
                        "scales differ by more than 2%")
    if scale["scale_cv"] > 0.02 or scale["scale_spread"] > 0.02:
        warnings.append("check for a rescaled print mixed into the scene, or a bent board")
    if scale["board_reproj_px"] > 2:
        warnings.append(f"board corners reproject at {scale['board_reproj_px']:.1f} px: is the board flat?")
    return failures, warnings

In [ ]:
registered, total = model.num_reg_images(), len(frames)
reproj = model.compute_mean_reprojection_error()
print(f"registered {registered} of {total} frames ({registered / total:.0%}); "
      f"mean reprojection error {reproj:.2f} px; stage time {(time.time() - T0) / 60:.1f} min")
failures, warnings = stage1_problems(registered, total, reproj, scale, REFERENCE)
for w in warnings:
    print("warning:", w)
assert not failures, "; ".join(failures)  # nothing is written when a gate fails
write_stage1(PROJECT / "stage1", frames, model, scale, heldout)  # refuses below 80% registered
if REFERENCE:
    (PROJECT / "stage1" / ".reference").touch()  # stage 2 then knows it runs on the reference
print("passed")

### Question 1. Poses up to scale

Using the projection equation and the reprojection error above, show that structure from motion cannot recover the size of a scene from images alone. Besides the scale, which other transformations of all cameras and points together leave every reprojection error unchanged, and how many degrees of freedom does that leave free? Point to the printed line in your run that converts model units to meters.

*Write your answer here.*

### Question 2. What the board measures

Explain how the triangulated board corners and your measured square fix the scale, and which mistakes in the board the per-frame check (`scale_cv`, `scale_spread`) can detect and which it cannot. Point to the printed values in your run that this check produced, or, on the reference object, to the line that says what set the scale instead.

*Write your answer here.*

### Question 3. How a scale error grows

Suppose the factor from model units to meters were 5% too large. Derive how a length, a volume and a moment of inertia computed with a fixed measured mass would change, and say which of them stage 5 is most sensitive to. Point to one length printed in your run and give the value it would show under that error.

*Write your answer here.*